# Soybean experiment analysis, quantity schema 2
Leaf metrics use only the `leaf` organ. Whole-plant quantities include every organ. Areas are geometric mesh areas, counted once. Leaf face-mean irradiance averages the two faces, whereas incident totals sum them. Scores use relative linear RGB luminance, not calibrated PAR.

This notebook uses the same functions and files as the report and slides. Select a study folder below. Old experiments remain in the legacy notebook and are not mixed with these incident totals.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
repo = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / 'Scripts/analyze_soybean_experiments.py').is_file())
sys.path.insert(0, str(repo / 'Scripts'))
from analyze_soybean_experiments import analyze_study
study = repo / 'out/soybean-experiments/20261016_robust_v1'
facts = analyze_study(study, plots=True)
display(pd.DataFrame({name: {'cases': value['cases'], 'plants': value['measured_plants'], 'unique_seeds': value['unique_generation_seeds'], 'checks': value['reconciliation']} for name, value in facts.items()}).T)

## Single-plant generation and realized geometry
Both variants use age 45 days, max 25 trifoliate nodes, internode max length 0.035 m, trifoliate leaf scale U(0.10, 0.14) m and petiole length U(0.12, 0.16) m. Broad aspect ratio is 1.0, narrow 0.3. Unifoliate scale stays at the Helios default 0.02 m. Longitudinal subdivision count is 8, but transverse counts differ (8 versus 4). Helios also adjusts stem girth to downstream leaf area. This is a complete-plant parameter comparison, not an equal-area or fixed-stem shape intervention.

In [ ]:
display(pd.read_csv(study / 'analysis/geometry_representative/plant_geometry.csv'))
library_geometry = pd.read_csv(study / 'analysis/geometry_library/geometry_summary.csv')
geometry_columns = ['asset_name'] + [metric + '_' + statistic for metric in ['leaf_count', 'height_m', 'leaf_mesh_area_m2', 'shoot_mesh_area_m2'] for statistic in ['mean', 'std']]
display(library_geometry[geometry_columns])
display(pd.read_csv(study / 'analysis/00_representative_pair/run_metrics.csv')[[
    'asset_name', 'leaf_mean_irradiance', 'leaf_incident_per_plant', 'whole_incident_per_plant',
    'leaf_area_per_plant_m2', 'whole_surface_area_per_plant_m2']])
single = facts['01_single_plant']
print(f"Repeated isolated-plant comparison: {single['paired_runs']} paired draws, {single['unique_generation_seeds']} unique generation seeds")
single_metrics = ['leaf_mean_irradiance', 'leaf_incident_per_plant', 'whole_incident_per_plant', 'leaf_area_per_plant_m2']
display(pd.DataFrame({metric: single['metrics'][metric] for metric in single_metrics}).T)
for filename in sorted((study / 'raw/00_representative_pair/captures').glob('*.png')):
    display(Image(filename=str(filename), width=800))

## Completed experiments and rankings
Choose one experiment ID. Field subset seed is the repeat unit, not the individual plant. Paired broad-minus-narrow differences hold spacing and subset seed fixed. Isolated plants have no meaningful allocated-ground-area score. A 256-sample comparison is a numerical reference, not an exact solution.

Rankings are exploratory. The independently generated holdout library is disjoint from the training library and checks candidate spacings selected from training results.

Within-field CV is the sample standard deviation of per-plant leaf face-mean irradiance divided by its arithmetic mean; it is not the standard error across field repeats. Receiver weights use geometric triangles without fractional texture-alpha correction. Static plants may interpenetrate. These relative incident-light results do not establish absorption, daily PAR, or crop yield.

In [ ]:
experiment = '03_field_search' if '03_field_search' in facts else '00_representative_pair'
folder = study / 'analysis' / experiment
ranked = pd.read_csv(folder / 'ranked_combinations.csv')
metrics = ['leaf_mean_irradiance', 'leaf_incident_per_plant', 'whole_incident_per_plant', 'leaf_incident_per_ground_area']
for metric in metrics:
    if ranked[metric].notna().any():
        print(metric)
        columns = ['asset_name', 'row_spacing_inches', 'column_spacing_inches', 'repeats', metric, metric + '_std']
        display(ranked.sort_values(metric, ascending=False).groupby('asset_name', sort=False).head(5)[columns])
display(pd.read_csv(folder / 'shape_comparison.csv'))
display(pd.read_csv(folder / 'paired_summary.csv'))
for filename in sorted(folder.glob('*.png')):
    display(Image(filename=str(filename), width=1100))

In [ ]:
if (study / 'analysis/sample_convergence.csv').exists():
    convergence = pd.read_csv(study / 'analysis/sample_convergence.csv')
    display(convergence.groupby(['samples','metric']).relative_difference_from_256.agg(['mean','min','max']))
for filename in ['holdout_validation.csv', 'buffer_sensitivity.csv', 'edge_control.csv', 'experiment_timings.csv', 'generation_timings.csv']:
    if (study / 'analysis' / filename).exists():
        print(filename)
        display(pd.read_csv(study / 'analysis' / filename))
display(pd.read_csv(study / 'analysis/stage_timings.csv')[[
    'stage','started_at_utc','completed_at_utc','elapsed_seconds','exit_code']])